# LLMCuisine-22: recipe generation on Kaggle GPUs
Group 44 · CGAS BIO544 mini-project · Deliverable D1. This notebook asks **one open-weight LLM** (Hugging Face weights)
for *traditional* recipes from the **22 CulinaryDB world regions** (5 recipes per call, plain and home-cook-persona prompts)
and logs every call (prompt, model revision, sampling settings, seed, raw output) for the dataset.

**Before you run (right-hand panel → Session options):**
1. **Accelerator → GPU T4 ×2** (or a GPU L4 option if your account shows one).
2. **Internet → On** (needs a phone-verified account).
3. **No token is needed for the planned models.** Qwen is ungated, and Llama loads from `RedHatAI/Llama-3.1-8B-Instruct`, a byte-identical copy of Meta's repo. Only the optional gated models (Gemma, Llama-3.2) need **Add-ons → Secrets → `HF_TOKEN`**.

**Run:** set `MODEL_KEY` and `MODE` in the next cell.
- **`MODE = "pilot"`**: 44 calls, about 15–30 min. Use **Run All** and watch the log.
- **`MODE = "full"`**: 704 calls, about 1–3 h. Use **Save Version → Save & Run All (Commit)** so it keeps running in the background.

**Output:** `/kaggle/working/llmcuisine22_<model>_<mode>.jsonl` (plus a `.gz` copy). Download it from the notebook's **Output** tab.
Results are saved after every batch. If a session dies, run the notebook again and it resumes where it stopped.


In [ ]:
# ==== SETTINGS: change these ====
MODEL_KEY = "qwen3.5-9b"   # core: "qwen3.5-9b", "llama-3.1-8b" | extra: "qwen3.5-4b", "ministral-3-8b" | optional: "olmo-3-7b", "phi-4-mini", "gemma-3-4b", "llama-3.2-3b", "exaone-3.5-7.8b"
MODE = "pilot"            # "pilot" = 44 calls (220 recipes) | "full" = 704 calls (3,520 recipes) | "sens" = 44 calls at T 1.0, top_p 1.0
MAX_CALLS = None             # e.g. 4 for a quick smoke test; None = every call in the mode
BATCH_SIZE = 64              # prompts per checkpoint (results are written after each batch)
FORCE_ENGINE = None          # None = vLLM with automatic fallback; or "transformers"
VLLM_VERSION = "0.34.0"     # pinned for reproducibility
NOTEBOOK_VERSION = "nb-v2"


In [ ]:
# Install vLLM BEFORE anything imports torch (2-5 min the first time). Falls back to transformers if it fails.
import importlib.util, subprocess, sys

def _pip(*args):
    print("pip install", *args, flush=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print((r.stdout or "")[-1500:], (r.stderr or "")[-3000:])
    return r.returncode == 0

VLLM_OK = False
if FORCE_ENGINE != "transformers":
    VLLM_OK = importlib.util.find_spec("vllm") is not None or _pip(f"vllm=={VLLM_VERSION}")
print("vLLM available:", VLLM_OK)


In [ ]:
# Embedded inputs (do not edit): prompt set + model registry. The sha256 check guarantees the exact prompts.
import base64, gzip, hashlib, json
PROMPT_SET_B64 = "H4sIAAAAAAACE+2d4XLbVrZm/89ToPhjKl2lm8ohQRK4NTVVkuLYTmLH13LSnYxvTUEULCGiADVIxq1098PMs8yLDUBJjg4Ftajh3uTZ1+tPypYpcZNaK6T04ezv771JVZ4U86Iqe/8e9S6nWVH29qLeeX7V/v0394/9b97+Y/nhf3z1VfsvzZ8nefNv5WI6bf5a56fN5/7vSXXSfrDX3Lr3x0en2XE+XX74Q11Msut/uWw+8FXzp9nVbJ5ftP/6c7WIsjqPsugsn15+WEyjSVWdF+VplM1mxWyelfMv289dzPK6vf3z4rc8usijYXRSfPiQ13k5j+Z1dv0wsml0fW9lVOeT4jKfffm+fF++zS+nV9HHYn4WfXv0w+uoKpu/flFWUTU/y+tonv9tHmXlSdR85CKrz0+qj+Wf9qKijPK/ZZN5c9v5WTGLZvN6MZkv6vzf35d/f9+7uYP3zVD/q/nrvJhP8/Yv73v/46SYnUVldpH/z/fN6O97zaOp85OiGfX65s1N/vhQNC3K2xt++eWX73v/ef0p13fXPKhPn9M8ZZfeDf/5n/98X747y6M70zRfbjaPLhbNf5rv7jy78yiGn56U6M91Mc+j/Le8vopWRmkebB59zJqPz6OP1WJ6EmWXl3lWt89HdvMV9q6fzGI+i/66aL5DxfxqL/pQ1e1dXVxO24H60WRxOYuOs9lFNi+i5pvSfFbdPKz8pBn/ZoD2ru4+0uZ7Hs3OqnoetY911t7lLQ1VfZLXX0Y/zvLoWXnaPMizJRbNI5jd8Pub6/3zv/39KUw7MaYdTMN0EEz3xZjuwzRMB8H0QIzpAUzDdBBMx2JMxzAN00EwPRRjegjTMB0E0yMxpkcwDdNBMD0WY3oM0zAdBNOJGNMJTMN0EEynYkynMA3TITDt5HJER5AI1GFALRgkkiQCdRhQyyWJjigRqMOAWi5KdGSJQB0G1HJZoiNMBOowoJYLEx1pIlBvDermnxssOrC+/oeVK6l77xZlMSuy3o4up75z+6phvL1x9PGsippv0Mdocdk+HzcT7i2BvWpuP5sX0+uvu3w6P1TVSVR9aP+pjj5kF0XzjW9vej34zbf8JLv6Mjo6a+/qIY9ahvNZ8yWz+fJuGoQW2bT5Yst7aj7YTohd2PUv7fJ+FdN7VdXVZFL1dnRh9xp23UyIXdhlwC7vd0K9Z813urrc5MWrr63X7Yj4hV8G/PJ+PdV7XZzm9SZ6DbT1upkQu7DLgF3e78l6z8+ycgO3Ym23lvNhFmYZMMv7ZV3vKC/z02za29Hl/2u4dTMhdmGXAbu8Mwi97/LyaoPXrZG2W8v5MAuzDJg19l+3qkXzuO4Ks/0zEeu8eN0ZE8/wzIBniWzmlZB5YRd23dqVymZeKZkXdmHXjV3+iRWBzMt9ReiFYAj2STAnG3o5R+qFXuh1q1dfMvVyfWIv1EKta7UGsrGXG5B7oRd63eoVS+ZeLib4Qi3UulZrqBB8uSHJF6J99kcYX//ylM6M1790qtU8v3U2LbLov0e3t9hCd8btvZZLXF833v2SZ9NreTjcCO7/GncnjrsDd3APFfe+OO59cAf3UHEfiOM+AHdwDxX3WBz3GNzBPVTch+K4D8Ed3EPFfSSO+wjcwT1U3MfiuI/BHdxDxT0Rxz0Bd3APFfdUHPcU3ME9UNydfKrqiFXhPVjeFWJVclV4D5Z3+VzVEazCe7C8ywerjmQV3oPlXT5ZdUSr8B4s7/LRqiNbhffgjpe0xHd3nPw5b+64Lv8Aq7fjC+HXOGpyb2bOm2CeAfP8ZRrv2nHLFl5B99T3a3RMjX3YZ8A+f9fG/mJy3r5t2rv7Hqq342MCa/jXOTcGYqABAwcr7zynzXNwOq9KYQcH+u8/OyfHQiw0YKG/uePwrG6evsnZop6cCXuovtTjwdkxERMNmLiy6OPqpMyvRH8a1F/6sToz5mGeAfP81pZX+fS4wbLMReVTL3PpGhv/8M+Af363y38s8rycXb9xExRQveylc24MxEADBibaKWBCCoh5mHffvFQ/BUxJAbEP+zrsW2mMUUoB9WtkiAFR0KqCbjsxoH7XDDkgGtrVsL+tHFC/l4YgEBUtqzjQDgK30F5DEoh6BtWLt5AE6tfbEAUioFEBh9uIAvXrb8gCUdDMCdiDty+f0IPT3LrLtYNm/Pab/HI2zWfbqsG5vdMW1Zd1+yeOvQL5Y5A7acgdkAN5YJD3pSHvAzmQBwb5QBryAZADeWCQx9KQx0AO5IFBPpSGfAjkQB4Y5CNpyEdADuSBQT6WhnwM5EAeGOSJNOQJkAN5YJCn0pCnQA7kYUHuxBNPR+QJ5aFRLh95knlCeWiUi2eejtATykOjXDz0dKSeUB4a5eKppyP2hPLQKBePPR25J5QHcrBjyXl3z8zLOn/gNPE2Ly9f4yDHzaAc3UAyA5L5uzReN/d91u5Ek7JNfYXG6sRoh3YGtPN3Z/xc1eezs6JuZm3vRUI89ZUZ92dGPdQzoJ6/K+P75pPb9UtS3qnvyVgZGOmQzoB0K50xVV1+zNpfXUhpp18Vszoy4iGeAfH87RjfZ+UkE36nqb4Zo2No5EM+A/L5LTFHk2ouYpx6McztpGiGZgY088tg/px9kmVnR0jW2f3bjolgCGZAsEQliUtI4pAMybprXcSTuJQkDu3QblW7lT4XhSROv8uFKA73TLrnVKM4/f4Wsjiss2ddXzmL20JdC2Ec5hk0b6Aexun3tJDGYZ9N+2KdNE6/nIU4Ds8MeTZUiOP0+1fI4zAs2POfh/uvn9C30ty6S63DrMxOsm0VrSzvrchKjnsC9aNQOzGoHVADdRhQ98Wg7gM1UIcB9UAM6gFQA3UYUMdiUMdADdRhQD0Ug3oI1EAdBtQjMahHQA3UYUA9FoN6DNRAHQbUiRjUCVADdRhQp2JQp0AN1EFA7eQSRUekCNWBUC0YKZIpQnUgVMtlio5QEaoDoVouVHSkilAdCNVyqaIjVoTqQKiWixUduSJU7/bcwpLr7r6R1/nHD9WiPLk+eHOX3e1fa73O9pWOcTnNgHcGvPO3QPzHIj/OJ5sbp778wR8U13DNgGv+7ocfmmesLqrNZVPf+LAyKbZhmwHbBisr/X7Llme5i2xz4wb66/zuTYt1WGfAunj157jooF6Us4/F5Hxz7+It/CB3f17MwzwD5vmbH15lZTGvjgVe7NSXP6yOim/4ZsA3v19kf3qc13MB3dT7RVYmxTZsM2CbXzNyu6josJouLo4lfqRTrxx5aGT8wz8D/iWK+VxCPod3eNfpXaqSz6Xkc7iGa75rK20kYvmcfgcJAR262dPN6QV0+u0jJHRoZ1O7vmZCp18/QkSHelbVGyhFdPrFI2R0CGdQuFgno9NvHSGkQzd7ug2VQzr9IhJSOgQ0dTb04EndJAcPaFYXx8d5Vm6vnuTmDjkeCtiPgu0kwXaADdiBgN2XBLsP2IAdCNgDSbAHgA3YgYAdS4IdAzZgBwL2UBLsIWADdiBgjyTBHgE2YAcC9lgS7DFgA3YgYCeSYCeADdiBgJ1Kgp0CNmCHAbYTTR4d0SNkh0K2bPRI9gjZoZAtmj06wkfIDoVs0fDRkT5Cdihki6aPjvgRskMhWzR+dOSPkL3rYxEHD3abvFnk9byK3haTqre7S7TXOAtxZ1COPyCaAdH8jRIvsuZTe7u7WHwNxZYjIhdyGZDL3xvRDvR1dVGUxaR5J9RwszieFpPe7q5gX8O27pnRD/0M6OfvjjjI6uPspJr1dndl/TqnaW+mRDEUM6BYfO8V7iA7yy6yzSyLt/GydjMooiGaAdH8PRHfNuQ278h6uztysIZkN0MiGIIZEMzvJnnXTFicZCdLHN5Vx9npZr95VK8o6RgY8RDPgHh+Tcnh4nizlzX1VpJ2QtRCLQNqJSopWkKKhmiIdvBg5cjmKVpKioZcyNXVMaKQoum3jRCj4Z9Z/5x8jKZfNkKOhmOGHOur5GiuT5CGaZh28GChiESQpl8lQpKGYXYMi1WTNP0iEaI0zLNp3lA2StPvDiFLw61gz1q+eFJPyItuq86KMttaR0hzZ3nzuDlkCdKPIe2kkHYgDdJBIN2XQroP0iAdBNIDKaQHIA3SQSAdSyEdgzRIB4H0UArpIUiDdBBIj6SQHoE0SAeB9FgK6TFIg3QQSCdSSCcgDdJBIJ1KIZ2CNEiHgLQTSw8d8SFMh8G0XHxIfgjTYTAtlh86AkSYDoNpsQDRkSDCdBhMiyWIjggRpsNgWixCdGSIML3bEwkvHuzx+LbIytPZYi+6A+3Wr6Je57ydNycHFFDNgGr+moZfzvJfW4o3dU19UcPKoMiGbAZk8/c1fLNoEd5UNfVlDd6YiIZoBkTz1zW8WJSbe6a+ruHulGiGZgY083c27Jdni2JTzdT3NNydEs3QzIBm/oKGo2Jyttj89Ux9S4M/J6qhmgHV/D6P582jOT2pNv99iHqRx+qk6IZuBnTzWzwa4koJ29TLPFYGRTZkMyBbopGqJaRqqIZqK6qlKqlaSqqGbMi2IttK04dMqqZf7kGshmnWTHPysZp+pwe5Gp4Z86wvn6vpl3oQrOGZMc8GGsGafrEHyRqumXMt1knW9Is9iNbwzaBvQ5VoTb/cg2wN2wycwvx6//DFE4o+2pt36dV+PDqsFuW8LvLZtio/nuf1Rfv+cb/5/tZFVi6xPfpYzGYc0QT4x4F38sA7gAf4YIHvywPfB3iADxb4gTzwA4AH+GCBj+WBjwEe4IMFfigP/BDgAT5Y4EfywI8AHuCDBX4sD/wY4AE+WOATeeATgAf4YIFP5YFPAR7gQwXeKSStjqgV4sMlXiNqJWuF+HCJV8haHWErxIdLvELY6khbIT5c4hXSVkfcCvHhEq8QtzryVogP7QzKNfPdRS4Hed3w1CA0/z2vpw0UvV1fKL/GeZR7Q3MiBfcsuOcv6zjIfsua/4vv3fxv/aq360v211FvZWbMwzwL5vVXX/WaJ0FSvP4WXvPujox3eGfBO3+dRzvR27Z48vpNm5x+6vs9uifHQiy0YKG/6OPoY3Ys+7ZTfeHHysh4h3cWvPMXfvzUPA/t9oyb3971dn26Yw3vVkbGO7yz4J1fGfPuqq6mktqpV8f4E2Md1lmwzm+O+WXRTHomnS2o98h0jI1/+GfBv0Q92UtI9nAP9zrcS7WTvZRkD/Mw75557ivlZM99RbSHeIh3Tzy3nWhPv42GbA8N7WrYV8729FtqCPcQz6B4A+VwT7+0hnQP8QyKF+ume/r1NcR7aGdPu+EW4j39LhvyPQQ0clb12bMntNk8e9bl2bOsGa8uo2eLurrMt9Vl499rVnI4FcQfQ9xJI+5AHMSDQrwvjXgfxEE8KMQH0ogPQBzEg0I8lkY8BnEQDwrxoTTiQxAH8aAQH0kjPgJxEA8K8bE04mMQB/GgEE+kEU9AHMSDQjyVRjwFcRAPCXEnnm464k0YD4tx+XiTfBPGw2JcPN90BJwwHhbj4gGnI+GE8bAYF084HREnjIfFuHjE6cg4YTyMYxst5d1tL0fT6rfsvPOU1DavH1/nTPDNpJzNwLLwLfNXYbytLrJSQDL1xRc3g+IYjoXvmL/n4k31wInDbV5Bv4Zi13NiGIaFb5i/0OLH87p5wvLebq/gX0Oxm0FxDMfCd8zfXfF20UC/+RtF9YUV13NiGIaFb9hwZR/oNKsXs95uTzCstQR0OSiO4Vj4jvn9Ki8W5WlWX/V2e4RiDcduBsUxHAvfsfG99bqHv+eTs6hBZnE8LSa93R7nWHOvrj8z5mFe+OYlOplZQmaGZVjW3ZgilZmlZGY4hmOd3ShCmZl+IQqhGYpZUcyphGb6pSekZkhmRrK+RmqmX2xCbIZiVhQbqMRm+h0m5GZIZkayWCU30y8sIThDMjOSDbWDM/2SEpIz1DNyovLozVNKSo7edIl2dHnzdbdRTtLcWdkSxQFKkH4MaSeFtANpkA4C6b4U0n2QBukgkB5IIT0AaZAOAulYCukYpEE6CKSHUkgPQRqkg0B6JIX0CKRBOgikx1JIj0EapINAOpFCOgFpkA4C6VQK6RSkQToEpJ1YeuiID2E6DKbl4kPyQ5gOg2mx/NARIMJ0GEyLBYiOBBGmw2BaLEF0RIgwHQbTYhGiI0OE6d0eWGip7i7peJWd1MXJXnSH2a1fRL3GGQVvTI4nIJoB0fxlDYdZC2u+qWnqqxr8OVEN1Qyo5q9s2J813+Aim23qmvrKhpVBkQ3ZDMjmL2/YL0+y6WK5eWQz29SXN6xOim7oZkC3eOVt5DybVuXmusX6byT9SdEN3Qzodn+hw0E2++sijw6rRTlv15Js5t1W1jl0joyACGhAQL/u43k2LSabv9qpt334c6IaqhlQzW/9+Cmb5qWAa+pVHyuDIhuyGZAtUcjdEnI3REM0X7RUI3dLyd1QDdV81VYqP6RyN/3OD4I3bLNnm9MJ3vTrP0je8M2gb32d5E2/CoToDd8M+jZQjt70+0HI3jDQsoGxRvam3xhC+IZr5lwbqoRv+nUhpG/YZuAk5zdv95/QFdLcukuub+qsnOTbKgv5pvnIhIOcIP040k4MaQfSIB0C0n0xpPsgDdIhID0QQ3oA0iAdAtKxGNIxSIN0CEgPxZAegjRIh4D0SAzpEUiDdAhIj8WQHoM0SIeAdCKGdALSIB0C0qkY0ilIg3QASDu59NARH8J0EEwLxofkhzAdBNNy+aEjQITpIJiWCxAdCSJMB8G0XILoiBBhOgim5SJER4YI07s8o7Ckursr5KAZqyGueTR3qd3+ddRrHFJYHZVTCvhmwDd/w8Prqr5oSBDwTX3Dw+qo+IZvBnxb6Q2ZzrJ20E1t0+8N8QbFNVwz4Jq/3eFgUZ8uRF7b1Jc6rI6Kb/hmwDd/l8P3V1W5uWvqmxzujolneGbAM3+Pw5usLmabi6a+xcGbE9MwzYBpq80gs+bzBd5AbqEbxJsU27DNgG1+OcibuvotLyV+O6JeD7I6Kr7hmwHfEqWsLSFrwzd8u+dbqpS1pWRt+IZvq76tdoUIZW1b6AohbEM2a7I5pbBNvyuEtA3hDArXV0jb9ItCiNsQzZhoA424Tb8ZhLwN1aypFuvkbdvoAyFwQzdrug2VAjf9ShASN4SzcGLz+dvDJ3SCNLfu0ut5nefb6wRp7+2c85oQ/SjRToxoB9EQHQDRfTGi+xAN0QEQPRAjegDREB0A0bEY0TFEQ3QARA/FiB5CNEQHQPRIjOgRREN0AESPxYgeQzREB0B0IkZ0AtEQHQDRqRjRKURD9O6JdnKZoSM0BOkQkBYMDUkNQToEpOVSQ0dsCNIhIC0XGzpyQ5AOAWm53NARHIJ0CEjLBYeO5BCkd3gsYQl1dwfIs8uiXsz2orvIbv+q6TXOJPiDciIB1wy45q9taJ7M2SybXm1um/rWhtVR8Q3fDPjmb21oBzq8mkyzk1zgFU59eUPXuHiHdwa885c4HNb5PN9cOPUdDt6cmIZpBkyLu95RVmVxXmwuXLyld5XeuHiHdwa885c57DcTlQLvKdU3OfiD4hquGXBtdO+nuDf5tLqsyjKfCbyvHG3jB7mOibEP+wzY5/eEHFb1h8XmzqmXhHhzYhqmGTAtUUniEpI4XMO1FddSpSQuJYnDN3xb9W2lHUQ2idPvCCGKQzyj4jmNKE6/JoQsDtWsqdZXzOL0u0II4xDPqHgDlTBOvzSENA7ZzMkW66Zx+u0hxHHoZ1e/oUYcp18hQh6HasEf9nz5+ugpBSLtzbvsetncS1ZGR4vj9mlunrdyvq06kaNq0Tz5+7P2/r+4nmMvepOdL79U82PdQdY8Qe1vMc+avxzVRfR9Vp43t22+P6/zy2xa/Ilzo9jxuB1OyQ6HHdhh3o6+kh197MAO83YMlOwYYAd2mLcjVrIjxg7sMG/HUMmOIXZgh3k7Rkp2jLADO8zbMVayY4wd2GHejkTJjgQ7sMO8HamSHSl2YId1O5xWVu4Iy9HDvh5qYTlpOXrY10MrLXfE5ehhXw+tuNyRl6OHfT208nJHYI4e9vXQCswdiTl6WD1QdS1IdwHSLWS9IM57rHGy6nZgDlWhmwXd/IU0f/yPvBfEEZI1hPtjZJRDOQvK+YtpPr1d6gVxLGUN4z5NjHAIZ0E4fyHNm0X5a3a8Fy0l6gVx2mWd95V3p0Y8xLMgnr+c5vni16zO5sLmqS+o8cdGPdSzoJ6/mOa7vM6mmbB56ltqvKkRD/EsiOfXI71reYxeZycLYfnUa5LuTY6ACGhBQL8h6c9588Qd5OVpNhU2UL006f7oKIiCFhRMVDO8hAwP3dDtofYk+QwvJcNDOZQ7erBASTzD0+9QIsTDOFPGOf0QT79HiRQP88yZ199CiqdfqESMh3v23Bvox3j6xUrkeJhnzrx4OzmefsESQR4GmjRwuKUgT79uiSQPB40cVn23/5TKpXf7nbbNs+nVtjqW2jtr3eaAKUg/hrSTQtqBNEgHgXRfCuk+SIN0EEgPpJAegDRIB4F0LIV0DNIgHQTSQymkhyAN0kEgPZJCegTSIB0E0mMppMcgDdJBIJ1IIZ2ANEgHgXQqhXQK0iAdAtJOLD10xIcwHQbTcvEh+SFMh8G0WH7oCBBhOgymxQJER4II02EwLZYgOiJEmA6DabEI0ZEhwvRuTy60VHdXmDy7KBp6/u1tdZGdlu0puD/Y3frF1GucVegcl+MKiGdAvJUdEUV+ctE8bZsqp78Vwh8U2ZDNgGz+Wojm9SLfVDT1HRB3hkQyJDMgmb//4ae8XA66mWbq+x68MREN0QyIFq+sF2u+9MY/q8X668TujIloiGZANH+rw1ExKVp0NxNNfYGDNyaiIZoB0fwiktdZM/1sU9HUW0e8MREN0QyI5heOvFvMJlm58UuaeruIPyeqoZoB1RLFZC0hWUM8xOsWL1VJ1lKSNWRDthXZVspFJJI1/T4RojUss2WZU4jW9BtEyNYwzZppfYVsTb8whHAN06yZNlAI1/T7QUjXMM2aabFCuqbfBUK8hmnWTBtqxGv6nR/ka7gW/mnMb9+8fkLfR3PrLre+zS6zclt9H8s7y5sHzmlMmH6MaSfFtINpmA6D6b4U032YhukwmB5IMT2AaZgOg+lYiukYpmE6DKaHUkwPYRqmw2B6JMX0CKZhOgymx1JMj2EapsNgOpFiOoFpmA6D6VSK6RSmYToIpp1YjugIEoE6EKjlgkSSRKAOBGqxJNERJQJ1IFCLRYmOLBGoA4FaLEt0hIlAHQjUYmGiI00E6t2eWFhi3d3/8c3ifFGdZ3vRHWq3flX1GgcW/Dk5sIBqBlTzlzj80NxX9nFj1dSXOPhzohqqGVDN3+Lwoqir2VlxsbFs6nscVidFN3QzoNvKKoe8PMmKTV3T3+Rwd0xEQzQDovmbHN5V51fVpp6p73G4OyWaoZkBzfw1Dj/Mss1/JaK+w+HulGiGZgY081s/vruq5hu/mqmXftydEs3QzIBmfufHi+r8PCtONjZNvfRjZVBkQzYDsiUaeVpCnoZqqLaiWqqRp6XkaaiGar5qKy0fYnmaftUHgRq+GfTNKQRq+n0fJGqYZs20vnyipl/3QaSGZ8Y8G8hHavplH2RqeGbMs1g+U9Ov+iBUwzNjng1VQjX9qg9SNWwzcPTyux/ePqHro7l1l1zfVXWebavrY3lnJecuIfpRop0U0Q6iIToEovtSRPchGqJDIHogRfQAoiE6BKJjKaJjiIboEIgeShE9hGiIDoHokRTRI4iG6BCIHksRPYZoiA6B6ESK6ASiIToEolMpolOIhugAiHZimaEjNATpIJCWCw1JDUE6CKTFUkNHbAjSQSAtFhs6ckOQDgJpsdzQERyCdBBIiwWHjuQQpHd5JmEJdXeXx9dZfrrYi46qRfPw7nC79Sun1ziVcH9WDiagnAHl/EUNz7Py9GNVykinvq6ha1q0QzsD2vlbG55/bEj+Vei1Tn15Q9e0aId2BrQbrL7B/DWXerUbbOEt5r1p0Q7tDGgXr2wDax6xjHSx/kaw1VlRDuUMKOevdzhYzDKh1zn1BQ/3Z0U5lDOgnN8G8m3zXk3qRzr1UpCOYZEO6QxIN16RTkq5sb5yCIdw9oRLtDK6hIwO5VCuQ7lUL6NLyejQDu26tFvpDBHN6PR7Qwjp8M6od04vpNOvECGlwzuj3vW1Ujr9PhFiOpwz6dxAK6bT7xYhp8M5k87FajmdftEIQR3W2bRuqBTU6TeOkNRhnIkDoq+ePaFy5NWzLrleFScnzYN8ls3m2+oduXOXec1ZUeh+lG4nSreDbugOiO6+KN196IbugOgeiNI9gG7oDojuWJTuGLqhOyC6h6J0D6EbugOieyRK9wi6oTsguseidI+hG7oDojsRpTuBbugOiO5UlO4UuqE7HLqdbFbpCCvBOyS8hcNK0krwDglv2bTSEVeCd0h4y8aVjrwSvEPCWzavdASW4B0S3rKBpSOxBO8QzlW0gHdXpnzb3EtW9nZ4tfc65ymWQ3KIAr3C18vfRfEmm+YNrWXe2+EV52sY9mlOJEOy8CXzF0+8nNVZI88Or3pfw7DrIdELvcLXy98x8bLO/trb4UX368jVjIhaqBW+WvGqWhv+7BVvQS1+8kItA2r5+yLeLer2Zjs8cLCGXNdDohd6ha+XX1byfX6cldWGL17qLSU3UyIYgoUvmF9McnRVF1lvh2cu1lmi2c6IXMgVvlyJeOqVkHqhF3p1FY4IpV4pqReSIdkD9SISqZd+qwixF35Z8csJx1763SHkXrhlw62+cO6l3xBC8IVbNtwaiAdf+l0gJF/4ZcWvWD750u/9IPrCMDOGDaWjL/2OD7Iv7Ar4QONfnlTs8ZdOsfK/FZNqa6Ue7b1lnGKE6ceZdmJMO5iG6SCY7osx3YdpmA6C6YEY0wOYhukgmI7FmI5hGqaDYHooxvQQpmE6CKZHYkyPYBqmg2B6LMb0GKZhOgimEzGmE5iG6SCYTsWYTmEapkNg2snliI4gEajDgFowSCRJBOowoJZLEh1RIlCHAbVclOjIEoE6DKjlskRHmAjUYUAtFyY60kSg3vEZhb88VJxxDWt0uHxAd8nd/qXVa5xW6JiWswtYZ8A6fynDT3mdTerF75srp76bYXVUfMM3A775ixpeFZOzKpv83/9Tbm6c+saG+8PiHM4ZcM5f4HBUlVWdbe6b+g4Hf1BcwzUDrvnLHH7ImudKwDX1dQ7+oLiGawZc89c6vFnkx1MB19RXO/iD4hquGXDNr+f4eTHJ5iI/tal3dKyOim/4ZsA3v63j26z5yi27m+qm3tmxMim2YZsB2xK97C0he8M6rOuyLlXK3lKyN3zDt1XfVgo+JLM3/aIPwjekMymdUwnf9Ls/SN+QzZxsfZX0Tb8MhPgN2czJNlCJ3/S7QcjfkM2cbLFS/qbfFEIAh3AGhRvqBHD6zSEkcOhm4Ejn0f6rJ3SINLfusuuoWjSPf/8ibx5Xtq0qEe9OOd8J4Y8T7qQJdxAO4SER3pcmvA/hEB4S4QNpwgcQDuEhER5LEx5DOISHRPhQmvAhhEN4SISPpAkfQTiEh0T4WJrwMYRDeEiEJ9KEJxAO4SERnkoTnkI4hAdEuBPPNB2hJogHhbh8qEmqCeJBIS6eajpiTRAPCnHxWNORa4J4UIiL55qOYBPEg0JcPNh0JJsgHsKhjCXk3R0pb/J60dvtteLrHDdspuQQBnoZ0MvfVXFYTauL4yLr7fZi9TUUu50UzdDMgGb+lorDs2Ka93Z7ufw6jrVjIhiCGRBssLJYsMx/X+TTzV/IBvqLBW9GRTREMyCav5Xi2WSRnVR1b7fHBtbQ7GZQJEMyA5L5myh+rBeni+yqt9uTC2tIdjMokiGZAcn8fpODOvu9mPZ2e3ZiDceu50QxFDOgmF9psl+fNs9DUW7+U5l6p8mnUREN0QyIlsinZAkpGXqhV0dtiVhKlpKSoRmaPdBWIpOS6feUEJNhmBXDnFJMpl9OQk6GaZZM66vkZPq9JARlWGbHsoFKUKZfSEJShmV2LIs1kjL9GhKiMhwz49hQKSrTbx8hK8O0sA9NHr7++inFI83NOwWbNMQ0mP9WbK935I+7LKMvjj7mLWd70euq/tgMl5V70ddZufzQN0XZ/qF50qKXk3zaftrkTxyyRInHlXDCSjiUQAnbSvSFleijBErYVmIgrMQAJVDCthKxsBIxSqCEbSWGwkoMUQIlbCsxElZihBIoYVuJsbASY5RACdtKJMJKJCiBEraVSIWVSFECJUwr4aTTa0d8jRPGnRCPr8mvccK4E9L5tSPAxgnjTkgH2I4EGyeMOyGdYDsibJww7oR0hO3IsHHC5vmkpRXdRTwHeXvq55qz7hO32ztFsc5RQG9cDiqhnAXlVst5LvPyLFty/HVetjj0dnpUY60FZPdmRj7ksyCfv8Dl28W8fcckZp76HpfVgdEO7Sxo5290eZFPZ0V5Xix/eml57u302Mka3t2bGPEQz4J4/pKXm98Q9HZ6pGUN3W7mRDIksyCZv+XlaF5Nzs+q6cVe1P7iLi97Oz0us4Zu9yZGPMSzIJ5f9vO8RaI8XtSnUuap9/7cHxn1UM+Cen4J0A+zaSUVG6jXAN0dFt3QzYJuiWJOl5DToRzKrSqXaud0KTkd8iFfl3wrpUHSOZ1+fRBBHd5Z9M7pBnX6jUIkdZhn0ry+RlKn3yxEVIdlhiwb6EZ1+h1DZHWYZ9K8WDmr028eIqzDPZvuDdXCOv0iItI6fAv+pOmz/adUET3bf7jq61nWPMv7sy22EbV3m9/eaxl98VORz1vQZnn7O5RpcVmUjYAvy5OqzGfLs6evsml2df3Ho+aOs8uqztu/HGYXx9XJ8uPfZ1V7JPVgUbdfiAOpqPO4Ok5eHYc6qPMZqNOXV6ePOqjzGagzkFdngDqo8xmoE8urE6MO6nwG6gzl1RmiDup8BuqM5NUZoQ7qfAbqjOXVGaMO6nwG6iTy6iSogzqfgTqpvDop6qDOf311nMLVBI7LCXDnc3BH43ICrifAnc/BHYXrCRwXFODO5+COwgUFjisKcOdzcEfhigLHJQW48zm4o3BJgeOaAtz5r3xKrrWnuwrrFrTers/vrHFE7nZWjschmgHR/B1Dn/433tv1cZ91ti/cDotqqGZANX+p0O2bpN6uTwets7vyZlZEQzQDovl7hZofQWa9XZ8jWkOydk4EQzADgvnrg15dZe1a1d6uDxyt88PZ9ahohmYGNPM3Bd38Pq6368NJa2h2MyqaoZkBzfziqnagN2ft77svm6dC4p2jenHVyshoh3YGtPNLqz7lSr1dH4NaZ8/y7bCohmoGVEu00rSENA3REO2BmirJNC0lTUM1VHuolEowTdPvoyJOwzRDpjmNOE2/fIo8DcOMGNZXytP0G6cI1PDMjmcDpUBNv2mKRA3P7HgWKydq+vVSRGp4Z8+7oVqkpl8tRaaGa0Gf7nz34im9Us2tuwR7d5YVf3QJ6xdKtffHoUuQfhRpJ4i0A2mQ3j3SfUGk+yAN0rtHeiCI9ACkQXr3SMeCSMcgDdK7R3ooiPQQpEF690iPBJEegTRI7x7psSDSY5AG6d0jnQginYA0SO8e6VQQ6RSkQXrnSDvJ9NARH8J0AEyLxofkhzAdANOS+aEjQITpAJiWDBAdCSJMB8C0ZILoiBBhOgCmJSNER4YI07s7obCkurvX4yArT8+r873Ih3YXF1KvcULh3rScVMA5A875+xsOz4qG4+hVVshop77HoWtgzMM8A+b5ex1eztpGNAnn1Nc6+KNiG7YZsM3f7vDmbHGez2V8U1/vsDosxmGcAeP8PQ/7V4t5A9RVJiOd+o6HjnnxDu8MeOfveXjRYPOz1I9z6nse7k2LczhnwDm/JeTHk4bAhuBSSDv1jpCugTEP8wyYN+76/eVbqRe88ZZ+f/mW1zzMM2ZeopfWJaR1OIdz951LVdO6lLQO8zCvy7yVRhG5tE6/UIS4Dt3M6ebU4jr9bhHyOpQzqFxfM6/TLxshsEM8m+IN9AI7/e4REjuksyhdrJrY6XeQENmhnlH1hqqRnX4ZCZkd6lk5Nfrj0VN6SZpbd0n26cP6lST7F3nz5GVl9MWPZfOcnURH82yez/7EGVIIf5RwJ0O4g3AID5PwvgzhfQiH8DAJH8gQPoBwCA+T8FiG8BjCITxMwocyhA8hHMLDJHwkQ/gIwiE8TMLHMoSPIRzCwyQ8kSE8gXAID5PwVIbwFMIhPEjCnVCm6Qg1QTxQxKVCTVJNEA8UcaFU0xFrgnigiAvFmo5cE8QDRVwo13QEmyAeKOJCwaYj2QTxsA5oLCHv7mFpB3qTTYoPxSR63cxx9jGfzfciH/TtXUG+xgGNhyfmmAYCGhDQ337xugH756o+38Q59aUX3pBohmYGNOvfe5379NbpqFrMz3q7uLp+zdc3f1KEQzgDwg3uva61X355fPb//6VtsI2XtrtzIhuyGZAtvvfq9qo42fRnt3gbr21350Q2ZDMgm7/T4l3zAGabaKa+w+KPCREMwQwI5jesHGbTonkcZZFtYpl6r8rKmKiGagZU8ytVvq8WxazIyo1MU+9R8adENEQzIFqim7Ml5GwIiID/QsBUPGdLydnQDM08zdxXOjmb+4qgDeMwrsM4pxG0OUfShm3Yds+2vkbS5vpEbdiGbfdsG8hGbfoVKWRtGGbKsFgha9NvRCFswzV7rg3lwzb9BhTSNkzbsWn/D/cOYZ8bswgA"
PROMPT_SET_SHA256 = "468c7d7f4c79c146484117f4b74647cbf00a38cdca1caf9e7b3ce9f515490526"
PROMPT_SET_LABEL = "v1"
CUSTOM_PROMPTS = False  # True: run every embedded prompt (D3 steering sets); False: MODE selects from v1
MODELS = json.loads(r'''{
 "llama-3.1-8b": {
  "hf_id": "RedHatAI/Llama-3.1-8B-Instruct",
  "canonical_id": "meta-llama/Llama-3.1-8B-Instruct",
  "family": "Llama (Meta)",
  "tier": "core",
  "params_b": 8.0,
  "license": "Llama 3.1 Community License (\"Built with Llama\" attribution; the licence applies to any copy)",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "max_tokens": 6144
 },
 "qwen3.5-9b": {
  "hf_id": "Qwen/Qwen3.5-9B",
  "family": "Qwen (Alibaba)",
  "tier": "core",
  "params_b": 9.0,
  "license": "Apache-2.0",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "chat_template_kwargs": {
   "enable_thinking": false
  },
  "engine_args": {
   "language_model_only": true
  },
  "max_tokens": 6144
 },
 "ministral-3-8b": {
  "hf_id": "mistralai/Ministral-3-8B-Instruct-2512-BF16",
  "family": "Mistral (Mistral AI)",
  "tier": "extra",
  "params_b": 8.9,
  "license": "Apache-2.0",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "engine_args": {
   "language_model_only": true
  },
  "max_tokens": 6144
 },
 "olmo-3-7b": {
  "hf_id": "allenai/Olmo-3-7B-Instruct",
  "family": "OLMo (Ai2, fully open data)",
  "tier": "optional",
  "params_b": 7.3,
  "license": "Apache-2.0",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "max_tokens": 6144
 },
 "phi-4-mini": {
  "hf_id": "microsoft/Phi-4-mini-instruct",
  "family": "Phi (Microsoft)",
  "tier": "optional",
  "params_b": 3.8,
  "license": "MIT",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "max_tokens": 6144
 },
 "gemma-3-4b": {
  "hf_id": "google/gemma-3-4b-it",
  "family": "Gemma (Google)",
  "tier": "optional",
  "params_b": 4.3,
  "gated": true,
  "license": "Gemma Terms of Use",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "engine_args": {
   "language_model_only": true
  },
  "fp16_unsafe": true,
  "max_tokens": 6144
 },
 "qwen3.5-4b": {
  "hf_id": "Qwen/Qwen3.5-4B",
  "family": "Qwen (Alibaba)",
  "tier": "extra",
  "params_b": 4.7,
  "license": "Apache-2.0",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "chat_template_kwargs": {
   "enable_thinking": false
  },
  "engine_args": {
   "language_model_only": true
  },
  "max_tokens": 6144
 },
 "llama-3.2-3b": {
  "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
  "family": "Llama (Meta)",
  "tier": "optional",
  "params_b": 3.2,
  "gated": true,
  "license": "Llama 3.2 Community License (\"Built with Llama\" attribution)",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "max_tokens": 6144
 },
 "exaone-3.5-7.8b": {
  "hf_id": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
  "family": "EXAONE (LG AI Research)",
  "tier": "optional",
  "params_b": 7.8,
  "license": "EXAONE AI Model License 1.1 (non-commercial research use)",
  "sampling": {
   "temperature": 0.7,
   "top_p": 0.95
  },
  "engine_args": {
   "trust_remote_code": true
  },
  "max_tokens": 6144
 }
}''')
_raw = gzip.decompress(base64.b64decode(PROMPT_SET_B64))
assert hashlib.sha256(_raw).hexdigest() == PROMPT_SET_SHA256, 'prompt set corrupted'
PROMPT_SET = [json.loads(l) for l in _raw.decode().splitlines()]
print(len(PROMPT_SET), 'prompts loaded; models:', list(MODELS))


Shared helpers: an exact copy of `generation/common.py` from the project repository.

In [ ]:
"""Shared helpers for LLMCuisine-22 generation (D1).

Pure standard library on purpose: this file is used by the Mac-side API runner and is also
inlined verbatim into the Kaggle notebook, so both paths build identical prompts, call ids and
seeds, and parse outputs the same way.

Design (see docs/IMPLEMENTATION_PLAN.md, Phase 4):
* 22 regions x 2 conditions (plain, persona) x n_reps replicates. Each call asks for 5 recipes.
  The full run uses 16 replicates (704 calls, 3,520 recipes per model). The pilot uses replicate 0
  only (44 calls).
* Prompts are model-agnostic and matched across models (a paired design). The persona place
  rotates through the region's 8 places so each is used equally often.
* call_id and seed are deterministic functions of (model_key, prompt key).
"""

from __future__ import annotations

import hashlib
import json
import re

PROMPT_SET_VERSION = "v1"
CONDITIONS = ("plain", "persona")
N_REPS_FULL = 16
# Pre-registered decoding-sensitivity subset (plan §5, Phase 4 step 4), open-weight models only: replicate 0 of every
# region x condition again at a common T = 1.0, top_p = 1.0. Same prompts, call ids and seeds as the main run (a
# paired design); parse_outputs keeps it apart through its mode ("sens").
SENSITIVITY_SAMPLING = {"temperature": 1.0, "top_p": 1.0}
SENSITIVITY_REPS = 1


def prompt_key(version: str, region_code: str, condition: str, rep: int) -> str:
    return f"{version}|{region_code}|{condition}|{rep:02d}"


def build_prompt_set(
    regions: dict, templates: dict, n_reps: int = N_REPS_FULL, version: str = PROMPT_SET_VERSION
) -> list[dict]:
    """Build the model-agnostic prompt set.

    regions: {code: {"label", "phrase", "places"}}; templates: {"system", "format", "plain", "persona"}.
    """
    fmt = templates["format"].strip()
    system = templates["system"].strip()
    out = []
    for code in sorted(regions):
        reg = regions[code]
        places = list(reg["places"])
        # Region-specific rotation offset so rep 0 does not always use the first-listed place.
        offset = int(hashlib.sha256(code.encode()).hexdigest(), 16) % len(places)
        for cond in CONDITIONS:
            for rep in range(n_reps):
                place = places[(rep + offset) % len(places)] if cond == "persona" else None
                user = templates[cond].format(phrase=reg["phrase"], place=place, format=fmt).strip()
                out.append(
                    {
                        "key": prompt_key(version, code, cond, rep),
                        "version": version,
                        "region_code": code,
                        "region_label": reg["label"],
                        "condition": cond,
                        "rep": rep,
                        "place": place,
                        "system": system,
                        "user": user,
                    }
                )
    return out


def select_mode(prompt_set: list[dict], mode: str) -> list[dict]:
    if mode == "pilot":
        return [p for p in prompt_set if p["rep"] == 0]
    if mode == "full":
        return list(prompt_set)
    if mode == "sens":
        return [p for p in prompt_set if p["rep"] < SENSITIVITY_REPS]
    raise ValueError(f"unknown mode {mode!r} (use 'pilot', 'full' or 'sens')")


def sampling_for(model_sampling: dict, mode: str) -> dict:
    """The decoding parameters of a run: the model's own, or the common sensitivity setting in mode 'sens'."""
    return dict(SENSITIVITY_SAMPLING) if mode == "sens" else dict(model_sampling)


def call_id(model_key: str, key: str) -> str:
    return hashlib.sha256(f"{model_key}|{key}".encode()).hexdigest()[:16]


def seed_for(model_key: str, key: str) -> int:
    return int(hashlib.sha256(f"seed|{model_key}|{key}".encode()).hexdigest()[:8], 16) & 0x7FFFFFFF


# ---------------------------------------------------------------- output parsing
_THINK_BLOCK = re.compile(r"<think>.*?</think>", re.S | re.I)
_FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.S | re.I)
_TRAILING_COMMA = re.compile(r",\s*([}\]])")


def strip_reasoning(text: str) -> str:
    """Remove <think>...</think> blocks. If a closing tag exists without an opening tag, keep only what follows it."""
    t = _THINK_BLOCK.sub("", text or "")
    if "</think>" in t:
        t = t.split("</think>")[-1]
    return t.strip()


def _repair(s: str) -> str:
    s = _TRAILING_COMMA.sub(r"\1", s)
    return s.replace("“", '"').replace("”", '"')


_RECIPE_BOUNDARY = re.compile(r'\][\s\]}]*,\s*\{?\s*("title"\s*:)')


def normalize_recipe_boundaries(s: str) -> str:
    """Rewrite every boundary between two recipes to the canonical '...], {"title": ...' + closing brace.

    A recipe's last field is a list, so a boundary is ']' + '}' + ',' + '{"title":'. Slips seen in the logs:
    Qwen pilot '], {"title"' (missing '}'); Gemini full run ']}, "title"' (missing '{'), ']]}, {"title"' and
    ']}}, {"title"' (an extra closer), and ']}]}, {"title"' (the document closed after the first recipe).
    Only tried after strict parsing fails.
    """
    return _RECIPE_BOUNDARY.sub(r"]}, {\1", s)


def balance_brackets(s: str) -> str:
    """Fix mismatched or missing JSON closers without touching any text inside strings.

    Seen in the pilot: Gemini sometimes omits the ']' that closes the last instructions list ('..."}]}').
    When a closer does not match the innermost open bracket, the missing closers are inserted; a stray closer with
    nothing open is dropped; open brackets left at the end are closed.
    """
    pairs = {"{": "}", "[": "]"}
    out, stack, in_str, esc = [], [], False, False
    for ch in s:
        if in_str:
            out.append(ch)
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in pairs:
            stack.append(ch)
        elif ch in "]}":
            if ch not in (pairs[b] for b in stack):
                continue  # stray closer
            while stack and pairs[stack[-1]] != ch:
                out.append(pairs[stack.pop()])
            stack.pop()
        out.append(ch)
    if in_str:
        out.append('"')
    out.extend(pairs[b] for b in reversed(stack))
    return "".join(out)


def _recipe_stream(text: str) -> list | None:
    """Recipes from a reply written as several JSON values instead of one document.

    Seen in the Gemini full run (JSON mode): the "recipes" list and the top-level object are closed after the first
    recipe and the remaining recipes follow as bare objects: {"recipes": [r1]}, r2, r3, r4, r5]}. Every value is
    decoded in turn; recipe lists and recipe-like objects (with "ingredients") are collected; separators and stray
    closing brackets between values are skipped. Nothing inside a value is changed.
    """
    dec = json.JSONDecoder()
    i, items, n_values = text.find("{"), [], 0
    while 0 <= i < len(text):
        try:
            obj, i = dec.raw_decode(text, i)
        except json.JSONDecodeError:
            break
        n_values += 1
        if isinstance(obj, dict) and isinstance(obj.get("recipes"), list):
            items.extend(obj["recipes"])
        elif isinstance(obj, dict) and "ingredients" in obj:
            items.append(obj)
        while i < len(text) and text[i] in " \r\n\t,]}":
            i += 1
        if i < len(text) and text[i] != "{":
            break
    return items if n_values > 1 and items else None


def _salvage_recipes(text: str) -> list | None:
    """Recover complete recipe objects from truncated JSON (e.g. the output hit max_tokens mid-recipe)."""
    m = re.search(r'"recipes"\s*:\s*\[', text)
    if not m:
        return None
    dec = json.JSONDecoder()
    i, items = m.end(), []
    while i < len(text):
        while i < len(text) and text[i] in " \r\n\t,":
            i += 1
        if i >= len(text) or text[i] != "{":
            break
        try:
            obj, j = dec.raw_decode(text, i)
        except json.JSONDecodeError:
            break
        items.append(obj)
        i = j
    return items or None


def extract_json(text: str) -> tuple[object | None, str]:
    """Return (parsed_object, status). Status is ok | repaired | salvaged | empty | unparseable."""
    if not text or not text.strip():
        return None, "empty"
    t = strip_reasoning(text)
    candidates = [mm.group(1) for mm in _FENCE.finditer(t)] + [t]
    for c in candidates:
        start, end = c.find("{"), c.rfind("}")
        if start == -1 or end <= start:
            continue
        s = c[start : end + 1]
        attempts = (
            ("ok", s),
            ("repaired", _repair(s)),
            ("repaired", balance_brackets(_repair(s))),
            ("repaired", balance_brackets(normalize_recipe_boundaries(_repair(s)))),
        )
        for status, attempt in attempts:
            try:
                return json.loads(attempt), status
            except json.JSONDecodeError:
                continue
    streamed = _recipe_stream(t)
    if streamed:
        return {"recipes": streamed}, "repaired"
    salvaged = _salvage_recipes(t)
    if salvaged:
        return {"recipes": salvaged}, "salvaged"
    return None, "unparseable"


def _ingredient_line(item: object) -> str:
    if isinstance(item, str):
        return item.strip()
    if isinstance(item, dict):
        keys = ("quantity", "amount", "unit", "item", "ingredient", "name", "preparation", "note")
        return " ".join(str(item[k]).strip() for k in keys if item.get(k)).strip()
    return str(item).strip()


def validate_recipes(obj: object) -> tuple[list[dict], list[str]]:
    """Normalize a parsed reply into [{"title", "ingredients": [str], "instructions": [str]}] and list any issues."""
    issues: list[str] = []
    if isinstance(obj, dict):
        recs = obj.get("recipes")
        if recs is None and "ingredients" in obj:
            recs = [obj]
    elif isinstance(obj, list):
        recs = obj
    else:
        return [], ["reply is not a JSON object"]
    if not isinstance(recs, list):
        return [], ["no 'recipes' list"]
    good = []
    for i, r in enumerate(recs):
        if not isinstance(r, dict):
            issues.append(f"recipe {i}: not an object")
            continue
        title = str(r.get("title") or r.get("name") or r.get("dish") or "").strip()
        ings = r.get("ingredients")
        if isinstance(ings, str):
            ings = [x for x in ings.split("\n")]
        lines = [s for s in (_ingredient_line(x) for x in (ings or [])) if s] if isinstance(ings, list) else []
        steps = r.get("instructions") or r.get("steps") or r.get("method") or []
        if isinstance(steps, str):
            steps = re.split(r"\n+", steps)
        steps = [str(s).strip() for s in steps if str(s).strip()] if isinstance(steps, list) else []
        if not title or len(lines) < 2:
            issues.append(f"recipe {i}: missing title or fewer than 2 ingredients")
            continue
        good.append({"title": title, "ingredients": lines, "instructions": steps})
    if len(recs) != 5:
        issues.append(f"expected 5 recipes, got {len(recs)}")
    return good, issues


In [ ]:
import os, json, time, gzip, base64, hashlib, platform, datetime, subprocess

# Read GPU facts from nvidia-smi so the notebook process never starts CUDA itself (keeps vLLM's multi-GPU start clean).
def _gpus():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=True).stdout.strip().splitlines()
    except Exception as e:
        raise RuntimeError("No GPU visible. Settings -> Accelerator -> GPU T4 x2, then restart the session.") from e
    rows = [[x.strip() for x in line.split(",")] for line in out if line.strip()]
    return [r[0] for r in rows], [round(float(r[1]) / 1024, 1) for r in rows], tuple(int(x) for x in rows[0][2].split("."))

M = MODELS[MODEL_KEY]
GPU_NAMES, GPU_MEM_GB, CC = _gpus()
N_GPU = len(GPU_NAMES)
if CC >= (8, 0):
    DTYPE = "bfloat16"                                      # L4 / A100 / H100
elif M.get("fp16_unsafe"):
    DTYPE = "float32"                                       # e.g. Gemma overflows in fp16; T4 has no bf16
else:
    DTYPE = "float16"                                       # T4 (sm75)
BYTES = 4 if DTYPE == "float32" else 2
NEED_GB = 1.1 * BYTES * M["params_b"] + 4                   # weights + KV-cache headroom
if NEED_GB > sum(GPU_MEM_GB):
    raise RuntimeError(f"{MODEL_KEY} needs about {NEED_GB:.0f} GB in {DTYPE}, but the GPUs have {sum(GPU_MEM_GB):.0f} GB. Pick a larger accelerator.")
TP = 1 if GPU_MEM_GB[0] >= NEED_GB else N_GPU
print(f"GPUs: {GPU_NAMES} {GPU_MEM_GB} GB | compute capability {CC} | dtype {DTYPE} | tensor_parallel {TP}")
print(f"Model: {MODEL_KEY} -> {M['hf_id']} | mode {MODE} | python {platform.python_version()}")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("No HF_TOKEN secret attached (fine for Qwen/OLMo):", type(e).__name__)

from huggingface_hub import model_info
try:
    REVISION = model_info(M["hf_id"], token=os.environ.get("HF_TOKEN")).sha
except Exception as e:
    raise RuntimeError(
        f"Cannot access {M['hf_id']}: {e}. For gated models (Llama) accept the licence on its Hugging Face page "
        "and attach your HF_TOKEN secret."
    )
print("Pinned model revision:", REVISION)


In [ ]:
from transformers import AutoTokenizer
import transformers

TRUST = bool(M.get("engine_args", {}).get("trust_remote_code"))
tok = AutoTokenizer.from_pretrained(M["hf_id"], revision=REVISION, trust_remote_code=TRUST)
llm, hf_model, ENGINE, ENGINE_KW, ENGINE_NOTES = None, None, None, {}, []


def expose_libcuda_for_jit(shim="/tmp/llmc_libcuda"):
    """Make `-lcuda` linkable for FlashInfer's JIT kernels (vLLM builds FlashInfer's gemma_rmsnorm for Qwen3.5).

    Kaggle images ship only the driver's versioned libcuda.so.1 (no libcuda.so, no stub), so nb-v1 failed with
    "/usr/bin/ld: cannot find -lcuda" and fell back to the slow transformers engine. The c++ linker driver reads
    LIBRARY_PATH, which the vLLM worker processes (and their ninja builds) inherit.
    """
    import glob
    found = []
    for pat in ("/usr/local/cuda*/lib64/stubs/libcuda.so", "/usr/local/cuda*/targets/*/lib/stubs/libcuda.so",
                "/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/local/nvidia/lib64/libcuda.so.1", "/usr/lib64/libcuda.so.1"):
        found += sorted(glob.glob(pat))
    if not found:
        try:
            out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
            found = [ln.split("=>")[-1].strip() for ln in out.splitlines() if "libcuda.so" in ln and "=>" in ln]
        except Exception:
            pass
    if not found:
        return "libcuda not found; JIT kernels may fail to link"
    target = os.path.realpath(found[0])
    os.makedirs(shim, exist_ok=True)
    link = os.path.join(shim, "libcuda.so")
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(target, link)
    old = os.environ.get("LIBRARY_PATH")
    os.environ["LIBRARY_PATH"] = shim + (":" + old if old else "")
    return f"LIBRARY_PATH shim: libcuda.so -> {target}"


if VLLM_OK and FORCE_ENGINE != "transformers":
    ENGINE_NOTES.append(expose_libcuda_for_jit())
    os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")  # no FlashInfer sampler on sm75; PyTorch sampler is used
    print(ENGINE_NOTES[-1])
    try:
        import vllm
        from vllm import LLM, SamplingParams
        base = dict(model=M["hf_id"], revision=REVISION, dtype=DTYPE, tensor_parallel_size=TP,
                    max_model_len=8192, gpu_memory_utilization=0.90, seed=0)
        variants = [dict(base, **M.get("engine_args", {}))]
        if M.get("engine_args", {}).get("language_model_only"):
            variants.append(dict(base, limit_mm_per_prompt={"image": 0, "video": 0}))
        variants.append(base)
        last_err, tried_eager, i = None, False, 0
        while llm is None and i < len(variants):
            kw = variants[i]
            try:
                llm = LLM(**kw)
                ENGINE, ENGINE_KW = f"vllm-{vllm.__version__}", {k: v for k, v in kw.items() if k != "model"}
            except (TypeError, ValueError) as e:  # unsupported engine argument: try the next variant
                last_err, i = e, i + 1
                print("vLLM init variant failed:", repr(e)[:300])
            except Exception as e:  # e.g. CUDA-graph capture on T4: retry once without CUDA graphs
                if tried_eager:
                    raise
                last_err, tried_eager = e, True
                print("vLLM init failed, retrying once with enforce_eager=True:", repr(e)[:600])
                variants[i] = dict(kw, enforce_eager=True)
                ENGINE_NOTES.append("enforce_eager retry after: " + repr(e)[:200])
        if llm is None:
            raise last_err
    except Exception as e:
        print("vLLM could not start, falling back to transformers:", repr(e)[:800])
        print("If the fallback runs out of GPU memory: Run > Restart & clear outputs, set FORCE_ENGINE = 'transformers', run again.")
        llm = None

if llm is None:
    import torch
    from transformers import AutoModelForCausalLM
    try:
        hf_model = AutoModelForCausalLM.from_pretrained(M["hf_id"], revision=REVISION, trust_remote_code=TRUST,
                                                        torch_dtype=getattr(torch, DTYPE), device_map="auto")
    except Exception as e:
        print("AutoModelForCausalLM failed, trying AutoModelForImageTextToText:", repr(e)[:300])
        from transformers import AutoModelForImageTextToText
        hf_model = AutoModelForImageTextToText.from_pretrained(M["hf_id"], revision=REVISION, trust_remote_code=TRUST,
                                                               torch_dtype=getattr(torch, DTYPE), device_map="auto")
    hf_model.eval()
    ENGINE = f"transformers-{transformers.__version__}"
print("Engine:", ENGINE)


In [ ]:
SAMPLING = sampling_for(M["sampling"], MODE)   # MODE "sens": the common T 1.0 / top_p 1.0 sensitivity setting
print("Sampling:", SAMPLING)

def render(p):
    msgs = [{"role": "system", "content": p["system"]}, {"role": "user", "content": p["user"]}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **M.get("chat_template_kwargs", {}))

def sha16(s):
    return hashlib.sha256(s.encode()).hexdigest()[:16]

def run_vllm(batch):
    prompts = [render(p) for p in batch]
    sps = [SamplingParams(max_tokens=M["max_tokens"], seed=seed_for(MODEL_KEY, p["key"]), **SAMPLING) for p in batch]
    outs = llm.generate(prompts, sps, use_tqdm=True)
    return [dict(text=o.outputs[0].text, finish_reason=o.outputs[0].finish_reason,
                 n_prompt_tokens=len(o.prompt_token_ids or []), n_output_tokens=len(o.outputs[0].token_ids),
                 rendered_sha16=sha16(pr)) for pr, o in zip(prompts, outs)]

HF_SUB = 16   # transformers fallback: prompts per generate() call (nb-v1 used 8: 49 tok/s); halves on CUDA OOM

def run_hf(batch):
    global HF_SUB
    import torch
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    s, res, i = SAMPLING, [], 0
    while i < len(batch):
        chunk = batch[i:i + HF_SUB]
        prompts = [render(p) for p in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(hf_model.device)
        torch.manual_seed(seed_for(MODEL_KEY, chunk[0]["key"]))
        try:
            with torch.no_grad():
                gen = hf_model.generate(**enc, do_sample=True, temperature=s.get("temperature", 1.0),
                                        top_p=s.get("top_p", 1.0), top_k=s.get("top_k", 0),
                                        max_new_tokens=M["max_tokens"], pad_token_id=tok.pad_token_id)
        except torch.cuda.OutOfMemoryError:
            del enc
            torch.cuda.empty_cache()
            if len(chunk) == 1:
                raise
            HF_SUB = len(chunk) // 2
            print("CUDA out of memory: retrying this chunk with sub-batch", HF_SUB, flush=True)
            continue
        for j in range(len(chunk)):
            out_ids = gen[j, enc["input_ids"].shape[1]:]
            n_out = int((out_ids != tok.pad_token_id).sum())
            res.append(dict(text=tok.decode(out_ids, skip_special_tokens=True),
                            finish_reason="length" if n_out >= M["max_tokens"] else "stop",
                            n_prompt_tokens=int(enc["attention_mask"][j].sum()), n_output_tokens=n_out,
                            rendered_sha16=sha16(prompts[j]), engine_batch=len(chunk)))
        i += len(chunk)
    return res

utcnow = lambda: datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
OUT = f"/kaggle/working/llmcuisine22_{MODEL_KEY}_{MODE}.jsonl"
todo_all = (list(PROMPT_SET) if CUSTOM_PROMPTS else select_mode(PROMPT_SET, MODE))[: MAX_CALLS or None]
done = set()
if os.path.exists(OUT):
    with open(OUT) as fh:
        for line in fh:
            try:
                done.add(json.loads(line)["call_id"])
            except Exception:
                pass
todo = [p for p in todo_all if call_id(MODEL_KEY, p["key"]) not in done]
print(f"{len(todo_all)} calls in mode '{MODE}': {len(done)} already done, {len(todo)} to run -> {OUT}")

META = dict(model_key=MODEL_KEY, hf_id=M["hf_id"], canonical_id=M.get("canonical_id", M["hf_id"]), revision=REVISION, family=M["family"], engine=ENGINE,
            engine_args=ENGINE_KW, engine_notes=ENGINE_NOTES, dtype=DTYPE, tensor_parallel=TP, gpus=GPU_NAMES, sampling=SAMPLING, mode=MODE,
            sampling_note=("presence_penalty not applied (transformers)" if llm is None and "presence_penalty" in SAMPLING else ""),
            chat_template_kwargs=M.get("chat_template_kwargs", {}), max_tokens=M["max_tokens"],
            prompt_set_version=PROMPT_SET_LABEL, prompt_set_sha256=PROMPT_SET_SHA256, notebook_version=NOTEBOOK_VERSION)
t0, n_tok = time.time(), 0
with open(OUT, "a") as fh:
    for b in range(0, len(todo), BATCH_SIZE):
        batch = todo[b:b + BATCH_SIZE]
        started = utcnow()
        results = run_vllm(batch) if llm is not None else run_hf(batch)
        finished = utcnow()
        for p, r in zip(batch, results):
            rec = {"call_id": call_id(MODEL_KEY, p["key"]), "prompt_key": p["key"], "region_code": p["region_code"],
                   "region_label": p["region_label"], "condition": p["condition"], "rep": p["rep"], "place": p["place"],
                   "system": p["system"], "user": p["user"], "seed": seed_for(MODEL_KEY, p["key"]),
                   "output_text": r["text"], "finish_reason": r["finish_reason"], "n_prompt_tokens": r["n_prompt_tokens"],
                   "n_output_tokens": r["n_output_tokens"], "rendered_prompt_sha16": r["rendered_sha16"],
                   "engine_batch": r.get("engine_batch"), "batch_started_utc": started, "batch_finished_utc": finished,
                   **({"meta": p["meta"]} if "meta" in p else {}), **META}
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_tok += r["n_output_tokens"]
        fh.flush()
        el = time.time() - t0
        n_done = b + len(batch)
        print(f"[{utcnow()}] {n_done}/{len(todo)} calls | {n_tok / max(el, 1):.0f} output tok/s | "
              f"ETA {el / n_done * (len(todo) - n_done) / 60:.1f} min", flush=True)
print("Generation finished in", round((time.time() - t0) / 60, 1), "min")


In [ ]:
from collections import Counter
rows = [json.loads(l) for l in open(OUT)]
status, n_rec, n5, n_ing, fin = Counter(), 0, 0, [], Counter(r["finish_reason"] for r in rows)
for r in rows:
    obj, st = extract_json(r["output_text"])
    status[st] += 1
    recs, issues = validate_recipes(obj) if obj is not None else ([], ["unparsed"])
    n_rec += len(recs)
    n5 += len(recs) == 5
    n_ing += [len(x["ingredients"]) for x in recs]
print(f"calls: {len(rows)} | parse status: {dict(status)} | finish: {dict(fin)}")
print(f"recipes parsed: {n_rec} (calls with exactly 5: {n5}/{len(rows)}) | mean ingredients/recipe: {sum(n_ing) / max(len(n_ing), 1):.1f}")
example, _ = extract_json(rows[0]["output_text"])
print(json.dumps((validate_recipes(example)[0] or [{}])[0], indent=1, ensure_ascii=False)[:1500])
with open(OUT, "rb") as src, gzip.open(OUT + ".gz", "wb") as dst:
    dst.write(src.read())
print("Saved:", OUT, "and", OUT + ".gz", "-> download from the Output tab")
if M["family"].startswith("Llama"):
    print("Reminder: Llama outputs must carry the 'Built with Llama' attribution in the dataset release.")
